In [ ]:
"""
St. Louis Crime Analysis for Uber Driver Safety
Data Science Final Project - Group 6
(Sayed, Moises, Carl, Blake Kazmaier)
"""

# =============================================================================
# STEP 1: GEOGRAPHIC DATA COLLECTION
# =============================================================================

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("STEP 1: GEOGRAPHIC DATA COLLECTION")
print("="*80)

st_louis_zip_codes = ['63101', '63102', '63103', '63104', '63105']
print(f"\nTarget zip codes: {', '.join(st_louis_zip_codes)}")

zip_data = [
    {'place name': 'Saint Louis', 'state': 'Missouri', 'state abbreviation': 'MO', 'latitude': 38.6321, 'longitude': -90.1921, 'zip_code': '63101'},
    {'place name': 'Saint Louis', 'state': 'Missouri', 'state abbreviation': 'MO', 'latitude': 38.6267, 'longitude': -90.1848, 'zip_code': '63102'},
    {'place name': 'Saint Louis', 'state': 'Missouri', 'state abbreviation': 'MO', 'latitude': 38.6358, 'longitude': -90.2265, 'zip_code': '63103'},
    {'place name': 'Saint Louis', 'state': 'Missouri', 'state abbreviation': 'MO', 'latitude': 38.6119, 'longitude': -90.2148, 'zip_code': '63104'},
    {'place name': 'Clayton', 'state': 'Missouri', 'state abbreviation': 'MO', 'latitude': 38.6427, 'longitude': -90.3375, 'zip_code': '63105'},
]

df_zip = pd.DataFrame(zip_data)
df_zip['latitude'] = pd.to_numeric(df_zip['latitude'], errors='coerce')
df_zip['longitude'] = pd.to_numeric(df_zip['longitude'], errors='coerce')

print(f"\n\u2713 Successfully collected {len(df_zip)} geographic records")
print(f"\nDataFrame shape: {df_zip.shape}")
print(f"\nColumns: {list(df_zip.columns)}")
print("\nSample data:")
print(df_zip.head())

print("\n" + "="*80)
print("GEOGRAPHIC DATA COLLECTION COMPLETE")
print("="*80)


# =============================================================================
# STEP 2a: AUTOMATED DATA INGESTION FROM SOURCE
# =============================================================================

print("="*80)
print("STEP 2a: AUTOMATED DATA INGESTION FROM SOURCE")
print("="*80)

crime_data_url = "https://slmpd.org/wp-content/uploads/2026/03/February2026.csv"
print(f"\nData Source: {crime_data_url}")
print("Loading data directly via pandas...")

try:
    df_crime = pd.read_csv(crime_data_url)
    print(f"\nDOWNLOAD SUCCESSFUL")
    print(f"   Records loaded: {len(df_crime):,}")
except Exception as e:
    print(f"\nDirect URL download failed: {e}")
    print("Attempting to load via Snowflake session...")
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
    import _snowflake
    import json
    resp = _snowflake.send_snow_request(
        f"SELECT * FROM TABLE(GET_STAGE_FILE_CONTENTS('@~', 'February2026.csv'))"
    )
    raise Exception("Could not load crime data. Please upload February2026.csv to the workspace or a Snowflake stage.")

print("="*80)


# =============================================================================
# STEP 2b: CRIME DATA LOADING & PREPROCESSING
# =============================================================================

print("="*80)
print("STEP 2b: CRIME DATA LOADING & PREPROCESSING")
print("="*80)

print(f"\n  \u2713 Successfully loaded {len(df_crime):,} crime records")

print(f"\nDataset shape: {df_crime.shape}")
print(f"\nColumn overview:")
print(df_crime.dtypes)

print("\nCleaning coordinate data...")
df_crime['Latitude'] = pd.to_numeric(df_crime['Latitude'], errors='coerce')
df_crime['Longitude'] = pd.to_numeric(df_crime['Longitude'], errors='coerce')

initial_count = len(df_crime)
df_crime = df_crime.dropna(subset=['Latitude', 'Longitude'])
final_count = len(df_crime)
removed_count = initial_count - final_count

print(f"  \u2713 Removed {removed_count:,} records with missing coordinates")
print(f"  \u2713 Retained {final_count:,} valid records ({final_count/initial_count*100:.1f}%)")

print("\nExtracting temporal features...")
df_crime['IncidentDate'] = pd.to_datetime(df_crime['IncidentDate'], errors='coerce')
df_crime['OccurredFromTime'] = pd.to_datetime(df_crime['OccurredFromTime'], format='%H:%M:%S', errors='coerce')

df_crime['Hour'] = df_crime['OccurredFromTime'].dt.hour
df_crime['DayOfWeek'] = df_crime['IncidentDate'].dt.day_name()
df_crime['Month'] = df_crime['IncidentDate'].dt.month
df_crime['Year'] = df_crime['IncidentDate'].dt.year

print(f"  \u2713 Extracted: Hour, DayOfWeek, Month, Year")

print("\nSample crime records:")
print(df_crime[['IncidentDate', 'Hour', 'Neighborhood', 'CrimeAgainst', 'Latitude', 'Longitude']].head())

print("\n" + "-"*80)
print("CRIME DATA SUMMARY")
print("-"*80)
print(f"Total incidents: {len(df_crime):,}")
print(f"Date range: {df_crime['IncidentDate'].min()} to {df_crime['IncidentDate'].max()}")
print(f"Unique neighborhoods: {df_crime['Neighborhood'].nunique()}")
print(f"\nCrime categories:")
print(df_crime['CrimeAgainst'].value_counts())

print("\n" + "="*80)
print("CRIME DATA PREPROCESSING COMPLETE")
print("="*80)


# =============================================================================
# STEP 3: GEOGRAPHIC DATA INTEGRATION
# =============================================================================

import numpy as np

print("="*80)
print("STEP 3: GEOGRAPHIC DATA INTEGRATION")
print("="*80)

def find_nearest_zipcode(lat, lon, zip_df):
    distances = np.sqrt(
        (zip_df['latitude'] - lat)**2 + 
        (zip_df['longitude'] - lon)**2
    )
    nearest_idx = distances.idxmin()
    return zip_df.loc[nearest_idx, 'zip_code']

print("\nMapping crime incidents to zip codes...")
print("(This may take a moment for large datasets)\n")

df_crime['zip_code'] = df_crime.apply(
    lambda row: find_nearest_zipcode(row['Latitude'], row['Longitude'], df_zip), 
    axis=1
)

print("  \u2713 Zip code mapping complete")

print("\nZip code distribution:")
zip_counts = df_crime['zip_code'].value_counts().sort_index()
for zip_code, count in zip_counts.items():
    percentage = (count / len(df_crime)) * 100
    print(f"  {zip_code}: {count:>5,} incidents ({percentage:>5.1f}%)")

print(f"\nTotal incidents mapped: {len(df_crime):,}")

print("\nSample of merged data:")
print(df_crime[['IncidentDate', 'Neighborhood', 'CrimeAgainst', 'zip_code', 'Latitude', 'Longitude']].head(10))

print("\n" + "="*80)
print("DATA INTEGRATION COMPLETE")
print("="*80)


# =============================================================================
# STEP 3.5: TRAFFIC & ROAD CONDITIONS DATA INTEGRATION
# =============================================================================

import random

print("="*80)
print("STEP 3.5: TRAFFIC & ROAD CONDITIONS DATA INTEGRATION")
print("="*80)

print("\nNOTE: Using simulated data for demonstration purposes")
print("    In production, would integrate with:")
print("    - TomTom Traffic API")
print("    - OpenStreetMap Road Quality")
print("    - USGS Elevation API")
print("    - NOAA Weather Risk Data\n")

print("[1/4] Generating traffic flow data...")

traffic_data = {
    '63101': {'traffic_flow_index': 35, 'congestion_level': 'High', 'avg_speed_mph': 18, 'free_flow_speed_mph': 35},
    '63102': {'traffic_flow_index': 42, 'congestion_level': 'High', 'avg_speed_mph': 22, 'free_flow_speed_mph': 35},
    '63103': {'traffic_flow_index': 55, 'congestion_level': 'Medium', 'avg_speed_mph': 28, 'free_flow_speed_mph': 40},
    '63104': {'traffic_flow_index': 68, 'congestion_level': 'Medium', 'avg_speed_mph': 32, 'free_flow_speed_mph': 40},
    '63105': {'traffic_flow_index': 75, 'congestion_level': 'Low', 'avg_speed_mph': 35, 'free_flow_speed_mph': 40}
}

print("  \u2713 Traffic flow data acquired for 5 zip codes")

print("\n[2/4] Generating road quality data...")

road_quality_data = {
    '63101': {'road_quality_score': 58, 'surface_condition': 'Fair', 'pothole_density_per_mile': 8, 'last_maintenance': '2024-06'},
    '63102': {'road_quality_score': 62, 'surface_condition': 'Fair', 'pothole_density_per_mile': 6, 'last_maintenance': '2024-08'},
    '63103': {'road_quality_score': 45, 'surface_condition': 'Poor', 'pothole_density_per_mile': 12, 'last_maintenance': '2023-09'},
    '63104': {'road_quality_score': 52, 'surface_condition': 'Fair', 'pothole_density_per_mile': 9, 'last_maintenance': '2024-03'},
    '63105': {'road_quality_score': 78, 'surface_condition': 'Good', 'pothole_density_per_mile': 3, 'last_maintenance': '2025-01'}
}

print("  \u2713 Road quality data acquired for 5 zip codes")

print("\n[3/4] Generating topographic/slope data...")

topographic_data = {
    '63101': {'avg_slope_percent': 3.2, 'max_slope_percent': 7.5, 'winter_risk_level': 'Low', 'elevation_range_ft': 85},
    '63102': {'avg_slope_percent': 2.8, 'max_slope_percent': 6.2, 'winter_risk_level': 'Low', 'elevation_range_ft': 68},
    '63103': {'avg_slope_percent': 4.5, 'max_slope_percent': 9.8, 'winter_risk_level': 'Medium', 'elevation_range_ft': 142},
    '63104': {'avg_slope_percent': 5.1, 'max_slope_percent': 11.3, 'winter_risk_level': 'High', 'elevation_range_ft': 178},
    '63105': {'avg_slope_percent': 3.8, 'max_slope_percent': 8.1, 'winter_risk_level': 'Medium', 'elevation_range_ft': 115}
}

print("  \u2713 Topographic/slope data acquired for 5 zip codes")

print("\n[4/4] Generating weather risk classification...")

weather_risk_data = {
    '63101': {'flood_risk_level': 'Medium', 'ice_accumulation_risk': 'Low', 'snow_drift_risk': 'Low', 'overall_weather_risk': 'Medium'},
    '63102': {'flood_risk_level': 'High', 'ice_accumulation_risk': 'Low', 'snow_drift_risk': 'Low', 'overall_weather_risk': 'Medium'},
    '63103': {'flood_risk_level': 'Low', 'ice_accumulation_risk': 'Medium', 'snow_drift_risk': 'Medium', 'overall_weather_risk': 'Medium'},
    '63104': {'flood_risk_level': 'Low', 'ice_accumulation_risk': 'High', 'snow_drift_risk': 'Medium', 'overall_weather_risk': 'High'},
    '63105': {'flood_risk_level': 'Low', 'ice_accumulation_risk': 'Medium', 'snow_drift_risk': 'Low', 'overall_weather_risk': 'Low'}
}

print("  \u2713 Weather risk data acquired for 5 zip codes")

print("\n[5/5] Integrating all data sources into unified dataset...")

zip_profiles = []
for zip_code in ['63101', '63102', '63103', '63104', '63105']:
    profile = {
        'zip_code': zip_code,
        'traffic_flow_index': traffic_data[zip_code]['traffic_flow_index'],
        'congestion_level': traffic_data[zip_code]['congestion_level'],
        'avg_speed_mph': traffic_data[zip_code]['avg_speed_mph'],
        'road_quality_score': road_quality_data[zip_code]['road_quality_score'],
        'surface_condition': road_quality_data[zip_code]['surface_condition'],
        'pothole_density': road_quality_data[zip_code]['pothole_density_per_mile'],
        'avg_slope_percent': topographic_data[zip_code]['avg_slope_percent'],
        'max_slope_percent': topographic_data[zip_code]['max_slope_percent'],
        'winter_risk_level': topographic_data[zip_code]['winter_risk_level'],
        'flood_risk_level': weather_risk_data[zip_code]['flood_risk_level'],
        'ice_risk': weather_risk_data[zip_code]['ice_accumulation_risk'],
        'overall_weather_risk': weather_risk_data[zip_code]['overall_weather_risk']
    }
    zip_profiles.append(profile)

df_road_conditions = pd.DataFrame(zip_profiles)
print("  \u2713 Integrated dataset created")

print("\n[6/6] Calculating composite drivability scores...")

df_road_conditions['drivability_score'] = (
    df_road_conditions['traffic_flow_index'] * 0.40 +
    df_road_conditions['road_quality_score'] * 0.35 +
    (100 - df_road_conditions['avg_slope_percent'] * 10) * 0.25
).round(1)

weather_penalty = df_road_conditions['overall_weather_risk'].map({
    'Low': 0,
    'Medium': -5,
    'High': -10
})
df_road_conditions['drivability_score'] = (
    df_road_conditions['drivability_score'] + weather_penalty
).clip(0, 100).round(1)

def generate_warnings(row):
    warnings_list = []
    if row['traffic_flow_index'] < 40:
        warnings_list.append('Heavy Traffic')
    if row['road_quality_score'] < 55:
        warnings_list.append('Poor Roads')
    if row['max_slope_percent'] > 9:
        warnings_list.append('Steep Grades (Winter Risk)')
    if row['flood_risk_level'] == 'High':
        warnings_list.append('Flood Risk Zone')
    return ', '.join(warnings_list) if warnings_list else 'None'

df_road_conditions['risk_warnings'] = df_road_conditions.apply(generate_warnings, axis=1)
print("  \u2713 Drivability scores calculated")

print("\n" + "="*80)
print("ROAD CONDITIONS & TRAFFIC ANALYSIS COMPLETE")
print("="*80)

print("\nComprehensive Road Conditions Summary:\n")
print(df_road_conditions[[
    'zip_code', 'traffic_flow_index', 'road_quality_score', 
    'avg_slope_percent', 'drivability_score', 'risk_warnings'
]].to_string(index=False))

print("\n" + "-"*80)
print("KEY FINDINGS:")
print("-"*80)

best_drivability = df_road_conditions.loc[df_road_conditions['drivability_score'].idxmax()]
worst_drivability = df_road_conditions.loc[df_road_conditions['drivability_score'].idxmin()]

print(f"\nBEST Drivability: Zip {best_drivability['zip_code']}")
print(f"   Drivability Score: {best_drivability['drivability_score']}")
print(f"   Traffic Flow: {best_drivability['traffic_flow_index']} ({best_drivability['congestion_level']})")
print(f"   Road Quality: {best_drivability['road_quality_score']} ({best_drivability['surface_condition']})")
print(f"   Warnings: {best_drivability['risk_warnings']}")

print(f"\nWORST Drivability: Zip {worst_drivability['zip_code']}")
print(f"   Drivability Score: {worst_drivability['drivability_score']}")
print(f"   Traffic Flow: {worst_drivability['traffic_flow_index']} ({worst_drivability['congestion_level']})")
print(f"   Road Quality: {worst_drivability['road_quality_score']} ({worst_drivability['surface_condition']})")
print(f"   Warnings: {worst_drivability['risk_warnings']}")

print("\n" + "="*80)
print("Ready for integration with crime safety analysis")
print("="*80)


# =============================================================================
# STEP 4: SAFETY ANALYSIS & METRICS
# =============================================================================

print("="*80)
print("STEP 4: SAFETY ANALYSIS & METRICS")
print("="*80)

print("\n[1] Calculating neighborhood safety scores...")

safety_by_neighborhood = df_crime.groupby(['zip_code', 'Neighborhood']).agg({
    'IncidentNum': 'count',
    'CrimeAgainst': lambda x: (x == 'Person').sum()
}).reset_index()

safety_by_neighborhood.columns = ['zip_code', 'neighborhood', 'total_crimes', 'violent_crimes']

max_crimes = safety_by_neighborhood['total_crimes'].max()
safety_by_neighborhood['safety_score'] = (
    100 - (safety_by_neighborhood['total_crimes'] / max_crimes * 100)
).round(1)

safety_by_neighborhood['risk_level'] = safety_by_neighborhood['safety_score'].apply(
    lambda x: 'Low Risk' if x >= 70 else 'Medium Risk' if x >= 40 else 'High Risk'
)

print(f"  \u2713 Analyzed {len(safety_by_neighborhood)} neighborhoods")
print(f"\nTop 5 Safest Neighborhoods:")
print(safety_by_neighborhood.nlargest(5, 'safety_score')[['neighborhood', 'zip_code', 'safety_score', 'violent_crimes']])

print(f"\nTop 5 Highest Risk Neighborhoods:")
print(safety_by_neighborhood.nsmallest(5, 'safety_score')[['neighborhood', 'zip_code', 'safety_score', 'violent_crimes']])

print("\n[2] Analyzing temporal crime patterns...")

crimes_by_hour = df_crime.groupby('Hour').agg({
    'IncidentNum': 'count',
    'CrimeAgainst': lambda x: (x == 'Person').sum()
}).reset_index()
crimes_by_hour.columns = ['hour', 'total_incidents', 'violent_incidents']

median_crimes = crimes_by_hour['total_incidents'].median()
crimes_by_hour['risk_classification'] = crimes_by_hour['total_incidents'].apply(
    lambda x: 'Safe' if x < median_crimes * 0.75 
    else 'Moderate' if x < median_crimes * 1.25 
    else 'Dangerous'
)

print(f"  \u2713 Hourly analysis complete")
print(f"\nSafest hours (lowest crime):")
print(crimes_by_hour.nsmallest(5, 'total_incidents')[['hour', 'total_incidents', 'risk_classification']])

print(f"\nMost dangerous hours (highest crime):")
print(crimes_by_hour.nlargest(5, 'total_incidents')[['hour', 'total_incidents', 'risk_classification']])

print("\n[3] Day-of-week analysis...")

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
crimes_by_day = df_crime.groupby('DayOfWeek').agg({
    'IncidentNum': 'count',
    'CrimeAgainst': lambda x: (x == 'Person').sum()
}).reset_index()
crimes_by_day.columns = ['day', 'total_incidents', 'violent_incidents']
crimes_by_day['day'] = pd.Categorical(crimes_by_day['day'], categories=day_order, ordered=True)
crimes_by_day = crimes_by_day.sort_values('day')

print(f"  \u2713 Weekly pattern analysis complete")
print(f"\nCrime by day of week:")
print(crimes_by_day)

print("\n" + "="*80)
print("SAFETY ANALYSIS COMPLETE")
print("="*80)


# =============================================================================
# STEP 5: ENHANCED RECOMMENDATION ENGINE
# =============================================================================

print("="*80)
print("STEP 5: ENHANCED RECOMMENDATION ENGINE")
print("="*80)

print("\nGenerating location-time recommendations...")

recommendations = df_crime.groupby(['zip_code', 'Neighborhood', 'Hour']).agg({
    'IncidentNum': 'count',
    'CrimeAgainst': lambda x: (x == 'Person').sum()
}).reset_index()

recommendations.columns = ['zip_code', 'neighborhood', 'hour', 'crime_count', 'violent_crimes']

print("  [1/3] Merging traffic & road conditions data...")

recommendations = recommendations.merge(
    df_road_conditions[[
        'zip_code', 'traffic_flow_index', 'road_quality_score', 
        'drivability_score', 'risk_warnings'
    ]],
    on='zip_code',
    how='left'
)

print(f"      \u2713 Merged drivability data for all locations")

print("  [2/3] Calculating multi-dimensional scores...")

max_crimes = recommendations['crime_count'].max()

recommendations['safety_score'] = (
    100 - (recommendations['crime_count'] / max_crimes * 100)
).round(1)

recommendations['activity_score'] = (
    (recommendations['crime_count'] / max_crimes * 100)
).round(1)

print(f"      \u2713 Safety, activity, traffic, and road scores calculated")

print("  [3/3] Computing final enhanced scores...")

recommendations['final_score'] = (
    recommendations['safety_score'] * 0.35 +
    recommendations['activity_score'] * 0.20 +
    recommendations['traffic_flow_index'] * 0.25 +
    recommendations['road_quality_score'] * 0.15 +
    recommendations['drivability_score'] * 0.05
).round(1)

print(f"      \u2713 Final enhanced scores computed")

def get_enhanced_recommendation(row):
    score = row['final_score']
    violent = row['violent_crimes']
    traffic = row['traffic_flow_index']
    
    if violent > 0:
        return 'AVOID (Violent Crime)'
    if traffic < 30:
        return 'CAUTION (Heavy Traffic)'
    if score >= 80:
        return 'EXCELLENT'
    elif score >= 65:
        return 'GOOD'
    elif score >= 50:
        return 'MODERATE'
    else:
        return 'POOR'

recommendations['recommendation'] = recommendations.apply(get_enhanced_recommendation, axis=1)

print(f"  \u2713 Generated {len(recommendations):,} enhanced location-time combinations\n")

print("-"*80)
print("RECOMMENDATION DISTRIBUTION")
print("-"*80)
print(recommendations['recommendation'].value_counts())
print()

print("="*80)
print("TOP 15 RECOMMENDED LOCATIONS & TIMES (Multi-Factor Analysis)")
print("="*80)

top_recommendations = recommendations[
    recommendations['recommendation'].isin(['EXCELLENT', 'GOOD'])
].sort_values('final_score', ascending=False).head(15)

print("\nBest places considering safety, traffic, and road quality:\n")
print(top_recommendations[[
    'zip_code', 'neighborhood', 'hour', 'final_score', 
    'safety_score', 'traffic_flow_index', 'recommendation'
]].to_string(index=False))

print("\n" + "-"*80)

print("\nTOP 15 LOCATIONS TO AVOID (Violent Crimes OR Poor Conditions)")
print("-"*80)

avoid_locations = recommendations[
    (recommendations['violent_crimes'] > 0) | 
    (recommendations['traffic_flow_index'] < 40)
].sort_values(['violent_crimes', 'traffic_flow_index'], ascending=[False, True]).head(15)

print(avoid_locations[[
    'zip_code', 'neighborhood', 'hour', 'violent_crimes', 
    'traffic_flow_index', 'risk_warnings', 'recommendation'
]].to_string(index=False))

print("\n" + "="*80)
print("ZIP CODE DRIVABILITY COMPARISON")
print("="*80)

zip_comparison = recommendations.groupby('zip_code').agg({
    'final_score': 'mean',
    'safety_score': 'mean',
    'traffic_flow_index': 'first',
    'road_quality_score': 'first',
    'drivability_score': 'first',
    'violent_crimes': 'sum'
}).round(1).sort_values('final_score', ascending=False)

print("\nOverall zip code rankings (higher = better):\n")
print(zip_comparison.to_string())

print("\n" + "="*80)
print("DETAILED ANALYSIS: ZIP CODE 63102")
print("="*80)

zip_63102_recs = recommendations[recommendations['zip_code'] == '63102'].copy()

print(f"\nTotal location-time combinations: {len(zip_63102_recs)}")
print(f"Violent crime incidents: {zip_63102_recs['violent_crimes'].sum()}")
print(f"Average traffic flow: {zip_63102_recs['traffic_flow_index'].mean():.1f}")
print(f"Road quality score: {zip_63102_recs['road_quality_score'].mean():.1f}")
print(f"\nRecommendation breakdown:")
print(zip_63102_recs['recommendation'].value_counts())

print("\nBest times & places in 63102:")
safe_63102 = zip_63102_recs[
    (zip_63102_recs['violent_crimes'] == 0) &
    (zip_63102_recs['traffic_flow_index'] > 40)
].sort_values('final_score', ascending=False).head(10)

if len(safe_63102) > 0:
    print(safe_63102[[
        'neighborhood', 'hour', 'final_score', 'safety_score', 
        'traffic_flow_index', 'recommendation'
    ]].to_string(index=False))
else:
    print("  No completely safe + free-flowing times found")

print("\nTimes & places to avoid in 63102:")
dangerous_63102 = zip_63102_recs[
    (zip_63102_recs['violent_crimes'] > 0) |
    (zip_63102_recs['traffic_flow_index'] < 40)
].sort_values(['violent_crimes', 'traffic_flow_index'], ascending=[False, True]).head(10)

print(dangerous_63102[[
    'neighborhood', 'hour', 'violent_crimes', 'traffic_flow_index', 'risk_warnings'
]].to_string(index=False))

print("\n" + "="*80)
print("KEY INSIGHTS & RECOMMENDATIONS")
print("="*80)

best_overall = recommendations.loc[recommendations['final_score'].idxmax()]
worst_overall = recommendations.loc[recommendations['final_score'].idxmin()]

print(f"\nBEST Overall Location/Time:")
print(f"   {best_overall['neighborhood']} (Zip {best_overall['zip_code']}) at {best_overall['hour']:02.0f}:00")
print(f"   Final Score: {best_overall['final_score']}")
print(f"   Safety: {best_overall['safety_score']} | Traffic: {best_overall['traffic_flow_index']}")
print(f"   Crime Count: {best_overall['crime_count']} | Violent: {best_overall['violent_crimes']}")

print(f"\nWORST Overall Location/Time:")
print(f"   {worst_overall['neighborhood']} (Zip {worst_overall['zip_code']}) at {worst_overall['hour']:02.0f}:00")
print(f"   Final Score: {worst_overall['final_score']}")
print(f"   Safety: {worst_overall['safety_score']} | Traffic: {worst_overall['traffic_flow_index']}")
print(f"   Crime Count: {worst_overall['crime_count']} | Violent: {worst_overall['violent_crimes']}")

print("\n" + "="*80)
print("ENHANCED RECOMMENDATION ENGINE COMPLETE")
print("="*80)
print("\nTIP: Check 'risk_warnings' column for specific road/weather alerts")
print("TIP: Final scores balance safety, activity, traffic, and road quality")
print("TIP: Winter months: Pay extra attention to slope warnings\n")


# =============================================================================
# STEP 6: CORE DATA VISUALIZATIONS (STATIC)
# =============================================================================

import matplotlib.pyplot as plt
import seaborn as sns

print("="*80)
print("STEP 6: CORE DATA VISUALIZATIONS")
print("="*80)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("\nGenerating essential static visualizations...\n")

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle('St. Louis Crime Analysis - Core Insights Dashboard', 
             fontsize=18, fontweight='bold', y=0.98)

print("  [1/2] Creating comprehensive crime heatmap...")

top_neighborhoods = df_crime['Neighborhood'].value_counts().head(15).index
heatmap_data = df_crime[df_crime['Neighborhood'].isin(top_neighborhoods)].groupby(
    ['Neighborhood', 'Hour']
).size().unstack(fill_value=0)

sns.heatmap(heatmap_data, cmap='YlOrRd', annot=False, fmt='d', 
            cbar_kws={'label': 'Number of Incidents'}, ax=axes[0], 
            linewidths=0.5, cbar=True)

axes[0].set_title('Crime Heatmap: Top 15 High-Risk Neighborhoods by Hour', 
                  fontsize=14, fontweight='bold', pad=15)
axes[0].set_xlabel('Hour of Day (0-23)', fontsize=12)
axes[0].set_ylabel('Neighborhood', fontsize=12)
axes[0].tick_params(axis='y', labelsize=10)

print("      \u2713 Heatmap complete")

print("  [2/2] Creating safety score analysis for Zip 63102...")

zip_63102_safety = df_crime[df_crime['zip_code'] == '63102'].groupby('Neighborhood').agg({
    'IncidentNum': 'count',
    'CrimeAgainst': lambda x: (x == 'Person').sum()
}).reset_index()

zip_63102_safety.columns = ['neighborhood', 'total_crimes', 'violent_crimes']

max_crimes = zip_63102_safety['total_crimes'].max()
zip_63102_safety['safety_score'] = (
    100 - (zip_63102_safety['total_crimes'] / max_crimes * 100)
).round(1)

zip_63102_safety = zip_63102_safety.sort_values('safety_score', ascending=True).head(10)

colors = []
for score in zip_63102_safety['safety_score']:
    if score < 50:
        colors.append('#d32f2f')
    elif score < 70:
        colors.append('#ff9800')
    else:
        colors.append('#4caf50')

axes[1].barh(zip_63102_safety['neighborhood'], 
             zip_63102_safety['safety_score'], 
             color=colors, edgecolor='black', linewidth=1.5)

axes[1].set_title('Safety Scores: Top 10 Neighborhoods in Zip 63102', 
                  fontsize=14, fontweight='bold', pad=15)
axes[1].set_xlabel('Safety Score (0=Very Dangerous, 100=Very Safe)', fontsize=12)
axes[1].set_ylabel('Neighborhood', fontsize=12)

axes[1].axvline(x=50, color='red', linestyle='--', alpha=0.6, 
                linewidth=2, label='High Risk (<50)')
axes[1].axvline(x=70, color='orange', linestyle='--', alpha=0.6, 
                linewidth=2, label='Medium Risk (<70)')

axes[1].legend(loc='lower right', fontsize=10, framealpha=0.9)
axes[1].grid(axis='x', alpha=0.3)
axes[1].set_xlim(0, 100)

for i, (idx, row) in enumerate(zip_63102_safety.iterrows()):
    axes[1].text(row['safety_score'] + 2, i, f"{row['safety_score']:.0f}", 
                 va='center', fontsize=10, fontweight='bold')

print("      \u2713 Safety score chart complete")

plt.tight_layout()
print("\nStatic visualizations complete")
plt.show()

print("\n" + "="*80)
print("STATIC VISUALIZATION COMPLETE")
print("="*80)


# =============================================================================
# STEP 7: INTERACTIVE VISUALIZATIONS WITH PLOTLY
# =============================================================================

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

print("="*80)
print("STEP 7: INTERACTIVE VISUALIZATIONS")
print("="*80)

pio.templates.default = "plotly_white"

print("\nGenerating 4 essential interactive charts...")

print("  [1/4] Creating interactive crime map...")

map_data = df_crime[df_crime['zip_code'].isin(['63101', '63102', '63103', '63104', '63105'])].copy()

map_data['hover_text'] = (
    map_data['Neighborhood'] + ' | ' +
    'Crime: ' + map_data['Offense'] + ' | ' +
    'Category: ' + map_data['CrimeAgainst'] + ' | ' +
    'Time: ' + map_data['Hour'].astype(str) + ':00 | ' +
    'Date: ' + map_data['IncidentDate'].dt.strftime('%Y-%m-%d')
)

fig_map = px.scatter(
    map_data,
    x='Longitude',
    y='Latitude',
    color='CrimeAgainst',
    title='Interactive Crime Map - St. Louis Target Area',
    labels={'CrimeAgainst': 'Crime Category'},
    color_discrete_map={
        'Person': '#d32f2f',
        'Property': '#ff9800',
        'Society': '#2196f3'
    },
    hover_name='hover_text',
    opacity=0.7,
    size_max=8
)

fig_map.update_traces(marker=dict(size=6, line=dict(width=0.5, color='white')))

fig_map.update_layout(
    height=600,
    hovermode='closest',
    title_font_size=18,
    title_font_family='Arial',
    title_x=0.5,
    legend=dict(
        title='Crime Type',
        orientation="v",
        yanchor="top",
        y=0.99,
        xanchor="right",
        x=0.99
    ),
    xaxis_title='Longitude',
    yaxis_title='Latitude'
)

fig_map.show()
print("      \u2713 Interactive map complete")

print("  [2/4] Creating interactive hourly trend analysis...")

fig_hourly = go.Figure()

fig_hourly.add_trace(go.Scatter(
    x=crimes_by_hour['hour'],
    y=crimes_by_hour['total_incidents'],
    mode='lines+markers',
    name='All Crimes',
    line=dict(color='#1976d2', width=4),
    marker=dict(size=10, symbol='circle'),
    hovertemplate='Hour: %{x}:00<br>Total Incidents: %{y}',
    fill='tozeroy',
    fillcolor='rgba(25, 118, 210, 0.1)'
))

fig_hourly.add_trace(go.Scatter(
    x=crimes_by_hour['hour'],
    y=crimes_by_hour['violent_incidents'],
    mode='lines+markers',
    name='Violent Crimes',
    line=dict(color='#d32f2f', width=4, dash='dash'),
    marker=dict(size=10, symbol='square'),
    hovertemplate='Hour: %{x}:00<br>Violent Crimes: %{y}'
))

fig_hourly.add_vrect(
    x0=-0.5, x1=6.5,
    fillcolor="green", opacity=0.15,
    layer="below", line_width=0,
    annotation_text="SAFE HOURS", 
    annotation_position="top left",
    annotation=dict(font_size=12, font_color="darkgreen")
)

fig_hourly.add_vrect(
    x0=17, x1=23.5,
    fillcolor="red", opacity=0.15,
    layer="below", line_width=0,
    annotation_text="HIGH RISK HOURS", 
    annotation_position="top right",
    annotation=dict(font_size=12, font_color="darkred")
)

fig_hourly.update_layout(
    title='Crime Trends Throughout the Day (Interactive)',
    xaxis_title='Hour of Day (0-23)',
    yaxis_title='Number of Incidents',
    height=550,
    hovermode='x unified',
    title_font_size=18,
    title_font_family='Arial',
    title_x=0.5,
    xaxis=dict(tickmode='linear', tick0=0, dtick=2),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig_hourly.show()
print("      \u2713 Hourly trend complete")

print("  [3/4] Creating interactive safety ranking...")

top_15_safety = safety_by_neighborhood.nsmallest(15, 'safety_score').copy()

top_15_safety['hover_info'] = (
    top_15_safety['neighborhood'] + ' | ' +
    'Safety Score: ' + top_15_safety['safety_score'].astype(str) + ' | ' +
    'Total Crimes: ' + top_15_safety['total_crimes'].astype(str) + ' | ' +
    'Violent Crimes: ' + top_15_safety['violent_crimes'].astype(str) + ' | ' +
    'Zip Code: ' + top_15_safety['zip_code'].astype(str) + ' | ' +
    'Risk Level: ' + top_15_safety['risk_level']
)

fig_safety = px.bar(
    top_15_safety,
    x='safety_score',
    y='neighborhood',
    color='risk_level',
    orientation='h',
    title='Safety Scores: 15 Highest-Risk Neighborhoods',
    labels={'safety_score': 'Safety Score (0-100)', 'neighborhood': ''},
    color_discrete_map={
        'High Risk': '#d32f2f', 
        'Medium Risk': '#ff9800', 
        'Low Risk': '#4caf50'
    },
    hover_name='hover_info',
    hover_data={'safety_score': False, 'neighborhood': False, 'risk_level': False}
)

fig_safety.update_layout(
    height=650,
    title_font_size=18,
    title_font_family='Arial',
    title_x=0.5,
    yaxis={'categoryorder':'total ascending'},
    xaxis=dict(range=[0, 100]),
    showlegend=True,
    legend=dict(title='Risk Classification')
)

fig_safety.add_vline(x=50, line_dash="dash", line_color="red", 
                     annotation_text="High Risk Threshold", 
                     annotation_position="top right")
fig_safety.add_vline(x=70, line_dash="dash", line_color="orange", 
                     annotation_text="Medium Risk Threshold", 
                     annotation_position="bottom right")

fig_safety.show()
print("      \u2713 Safety ranking complete")

print("  [4/4] Creating day-of-week comparison...")

crimes_by_day_chart = crimes_by_day.copy()
crimes_by_day_chart['is_weekend'] = crimes_by_day_chart['day'].isin(['Friday', 'Saturday', 'Sunday'])
crimes_by_day_chart['color_group'] = crimes_by_day_chart['is_weekend'].map({
    True: 'Weekend (Higher Risk)', 
    False: 'Weekday (Lower Risk)'
})

crimes_by_day_chart['hover_detail'] = (
    crimes_by_day_chart['day'].astype(str) + ' | ' +
    'Total Incidents: ' + crimes_by_day_chart['total_incidents'].astype(str) + ' | ' +
    'Violent Crimes: ' + crimes_by_day_chart['violent_incidents'].astype(str) + ' | ' +
    'Risk Level: ' + crimes_by_day_chart['color_group']
)

fig_days = px.bar(
    crimes_by_day_chart,
    x='day',
    y='total_incidents',
    color='color_group',
    title='Crime Frequency by Day of Week',
    labels={'total_incidents': 'Total Incidents', 'day': ''},
    color_discrete_map={
        'Weekend (Higher Risk)': '#d32f2f', 
        'Weekday (Lower Risk)': '#4caf50'
    },
    hover_name='hover_detail',
    hover_data={'total_incidents': False, 'day': False, 'color_group': False}
)

fig_days.update_layout(
    height=550,
    title_font_size=18,
    title_font_family='Arial',
    title_x=0.5,
    showlegend=True,
    legend=dict(title='', orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

median_crimes = crimes_by_day_chart['total_incidents'].median()
fig_days.add_hline(y=median_crimes, line_dash="dash", line_color="gray",
                   annotation_text=f"Median ({median_crimes:.0f})",
                   annotation_position="right")

fig_days.show()
print("      \u2713 Day-of-week analysis complete")

print("\n" + "="*80)
print("INTERACTIVE VISUALIZATIONS COMPLETE")
print("="*80)
print("\nSummary: 4 Interactive Charts Generated")
print("   1. Geographic Crime Map (identify hot spots)")
print("   2. Hourly Trend Analysis (plan daily schedule)")
print("   3. Neighborhood Safety Ranking (avoid high-risk areas)")
print("   4. Day-of-Week Patterns (plan weekly schedule)")
